In [9]:
import numpy as np
import pandas as pd

df = pd.read_csv("hantavirus2026.csv")

def clean_hantavirus_optimized(file_path: str) -> pd.DataFrame:
    # 1. Ingestión directa mapeando nulos desde el inicio
    df = pd.read_csv(file_path, na_values=["NA", "na", ""], keep_default_na=True)
    
    # 2. Renombrado masivo y vectorizado de columnas
    df.rename(columns={
        "cruise.crew..y.n.": "is_crew",
        "passenger..y.n.": "is_passenger",
        "Travel_flight": "travel_flight",
        "Gh_ID": "gh_id"
    }, inplace=True)
    
    # 3. Conversión masiva de fechas (Operación Vectorizada)
    date_cols = ["symptom_onset", "outcome_date", "treatment_date", "ship_boarded", "left_ship", "confirmation_date"]
    df[date_cols] = df[date_cols].apply(pd.to_datetime, errors="coerce")
    
    # 4. Optimización de Edad: Tratamiento de "70s" y numéricos sin bucles
    # Extrae los números. Si el string original contenía una 's', le suma 5 de forma condicional.
    age_clean = df["age"].astype(str).str.lower()
    has_s = age_clean.str.contains("s", na=False)
    df["age"] = pd.to_numeric(age_clean.str.extract(r"(\d+)", expand=False), errors="coerce")
    df.loc[has_s, "age"] += 5.0
    
    # 5. Normalización de texto y mapeo binario a Flags de BI (1.0 / 0.0)
    binary_map = {"y": 1.0, "n": 0.0, "yes": 1.0, "no": 0.0}
    df["is_crew"] = df["is_crew"].str.lower().str.strip().map(binary_map)
    df["is_passenger"] = df["is_passenger"].str.lower().str.strip().map(binary_map)
    df["confirmed"] = df["confirmed"].str.lower().str.strip().map(binary_map)
    
    # Estandarizar el resto de categóricos a minúsculas
    categorical_cols = ["status", "outcome", "sex", "nationality", "treatment"]
    df[categorical_cols] = df[categorical_cols].astype(str).apply(lambda x: x.str.lower().str.strip()).replace("nan", np.nan)

    # 6. Corrección de Typos vectorizada con reemplazo directo por diccionario
    typos_dict = {
        "diarreah": "diarrhea", 
        "diarrhiea": "diarrhea", 
        "gastrointesinal": "gastrointestinal",
        "flu like symptoms": "flu-like symptoms"
    }
    df["symptoms"] = df["symptoms"].astype(str).str.lower().str.strip()
    df["symptoms"].replace(typos_dict, regex=True, inplace=True)
    df["symptoms"].replace("nan", "unspecified", inplace=True)

    # 7. Desglose Atómico de Destinos (1FN) usando vectorización de strings
    # Limpia paréntesis, corrige Schiphol y toma el destino final después del 'then'
    travel_clean = df["travel_to"].astype(str).str.replace(r"\(.*?\)", "", regex=True).str.replace("Schipol", "Schiphol")
    final_destination = travel_clean.str.split("then").str[-1].str.strip()
    
    # Divide en Ciudad y País si existe la coma
    df["destination_city"] = final_destination.str.split(",").str[0].str.strip().str.title()
    df["destination_country"] = final_destination.str.split(",").str[1].str.strip().str.title()
    
    # Caso donde no había coma (ej. solo "Netherlands"): El valor queda en el país y limpiamos la ciudad
    only_country_mask = final_destination.str.contains(",") == False
    df.loc[only_country_mask, "destination_country"] = df.loc[only_country_mask, "destination_city"]
    df.loc[only_country_mask, "destination_city"] = np.nan
    
    # Eliminar columna sucia residual y formatear IDs
    df.drop(columns=["travel_to"], inplace=True)
    df["gh_id"] = df["gh_id"].astype(int)
    df["linked_id"] = pd.to_numeric(df["linked_id"], errors="coerce").astype("Int64")
    
    return df

# Ejecución rápida
if __name__ == "__main__":
    # df_limpio = clean_hantavirus_optimized("hantavirus2026.csv")
    print("Pipeline ultra-optimizado listo para ejecución masiva.")

Pipeline ultra-optimizado listo para ejecución masiva.


In [13]:
df.sample(20)

,Gh_ID,linked_id,status,linked_information,symptom_onset,outcome,outcome_date,treatment,treatment_date,cruise.crew..y.n.,...,Travel_flight,context,ship_boarded,left_ship,left_location,confirmed,confirmation_date,nationality,accession_id,sources
13,14,2.0,tested,on flight to johannsburg with women who later ...,NaN,NaN,NaN,monitored,NaN,n,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,spanish,NaN,https://www.reuters.com/business/healthcare-ph...
19,20,NaN,confirmed,evacuated,2026-05-14,NaN,NaN,quarantine,NaN,n,...,evacuation flight,from Yukon Terriroty. Isolating on Vancouver I...,2026-04-01,2026-05-10,Tenerife,y,2026-05-15,canadian,NaN,https://www.bbc.com/news/articles/c2k2p057ezno...
8,9,4.0,negative,NaN,NaN,NaN,NaN,hospitalised,2026-05-06,n,...,Medicalized aircraft 1,NaN,2026-04-01,2026-05-06,Cape Verde,NaN,NaN,german,NaN,https://www.bbc.com/news/live/cy592qeq071t; ht...
3,4,NaN,confirmed,NaN,2026-04-28,died,2026-05-02,NaN,NaN,n,...,NaN,NaN,2026-04-01,NaN,NaN,NaN,NaN,german,NaN,https://www.bbc.com/news/articles/c5y093d5n9ko
7,8,NaN,confirmed,NaN,2026-04-27,NaN,NaN,hospitalised,2026-05-07,y,...,Medicalized aircraft 2,NaN,2026-04-01,2026-05-06,Cape Verde,y,2026-05-06,dutch,PP_006W6RC.2,https://www.bbc.com/news/live/cy592qeq071t; ht...
10,11,2.0,negative,on flight to johannsburg with women who later ...,NaN,NaN,NaN,hospitalised,NaN,n,...,NaN,"arrived in Singapore May 6, 2026",2026-04-01,2026-04-24,St. Helena,NaN,NaN,singaporean permanent resident,NaN,https://www.straitstimes.com/singapore/2-singa...
12,13,2.0,monitored,on flight to johannsburg with women who later ...,NaN,NaN,NaN,monitored,NaN,n,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,french,NaN,https://www.france24.com/en/health/20260506-li...
11,12,NaN,probable,NaN,2026-04-28,NaN,NaN,NaN,NaN,n,...,NaN,NaN,2026-04-01,2026-04-14,Tristan de Cunha,NaN,NaN,british,NaN,https://ukhsa-newsroom.prgloo.com/news/uk-gove...
2,3,NaN,confirmed,NaN,2026-04-24,NaN,NaN,intensive care,2026-04-29,n,...,NaN,NaN,2026-04-01,2026-04-27,Ascension,y,2026-05-02,british,PP_00WANE.2,https://www.science.org/content/article/cruise...
4,5,NaN,confirmed,NaN,2026-05-01,NaN,NaN,intensive care,NaN,n,...,NaN,NaN,2026-04-01,2026-04-24,St. Helena,y,2026-05-06,swiss,PP_006WBLH.1,https://www.swissinfo.ch/eng/various/man-with-...


In [15]:
def impute_and_clean_dataset(file_path: str) -> pd.DataFrame:
    # 1. Leer el dataset convirtiendo vacíos y "NA" en objetos NaN reales
    df = pd.read_csv(file_path, na_values=["NA", "na", ""], keep_default_na=True)
    
    # 2. Renombrar columnas clave para estandarizar
    df.rename(columns={
        "cruise.crew..y.n.": "is_crew",
        "passenger..y.n.": "is_passenger",
        "Travel_flight": "travel_flight",
        "Gh_ID": "gh_id"
    }, inplace=True)
    
    # 3. Preprocesamiento rápido de la Edad (Tratamiento del texto "70s" a numérico)
    age_clean = df["age"].astype(str).str.lower()
    has_s = age_clean.str.contains("s", na=False)
    df["age"] = pd.to_numeric(age_clean.str.extract(r"(\d+)", expand=False), errors="coerce")
    df.loc[has_s, "age"] += 5.0  # Ajustar el rango "70s" a 75
    
    # Convertir otras columnas numéricas/flags binarias
    binary_map = {"y": 1.0, "n": 0.0, "yes": 1.0, "no": 0.0}
    for col in ["is_crew", "is_passenger", "confirmed"]:
        df[col] = df[col].astype(str).str.lower().str.strip().map(binary_map)

    # =============================================================================
    # ESTRATEGIA DE IMPUTACIÓN DE VALORES NaN
    # =============================================================================
    
    # --- A. Columnas Numéricas: Imputación con la MEDIA ---
    # Identificamos columnas numéricas (excluyendo el 'gh_id' por ser el identificador único)
    numeric_cols = ["age", "is_crew", "is_passenger", "confirmed"]
    
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            media_col = df[col].mean()
            # Si la columna es un Flag binario (0 o 1), redondeamos la media para mantener la coherencia
            if col in ["is_crew", "is_passenger", "confirmed"]:
                media_col = round(media_col)
            df[col].fillna(media_col, inplace=True)
            print(f"-> Columna numérica '{col}': NaN reemplazados con la media ({media_col:.2f})")

    # --- B. Columnas de Texto / Categóricas: Imputación con la MODA ---
    # Columnas de texto relevantes para el análisis
    text_cols = ["status", "outcome", "sex", "nationality", "symptoms", "travel_from", "left_location"]
    
    for col in text_cols:
        # Aseguramos formato limpio de texto en minúsculas y removemos nan en string residuales
        df[col] = df[col].astype(str).str.lower().str.strip().replace("nan", np.nan)
        
        if df[col].isnull().sum() > 0:
            # Calculamos la moda (el valor más común). [0] toma el primer resultado si hay empate.
            moda_col = df[col].mode()[0] if not df[col].mode().empty else "unspecified"
            df[col].fillna(moda_col, inplace=True)
            print(f"-> Columna de texto '{col}': NaN reemplazados con la moda ('{moda_col}')")

    # --- C. Columnas de Fechas: Tratamiento Especial ---
    # En fechas no se puede aplicar una media aritmética tradicional de manera directa sin romper la lógica temporal. 
    # Lo óptimo en salud es rellenar con una "Fecha de Control/Desconocida" o mantenerlas si representan hitos no ocurridos.
    date_cols = ["symptom_onset", "outcome_date", "treatment_date", "ship_boarded", "left_ship", "confirmation_date"]
    df[date_cols] = df[date_cols].apply(pd.to_datetime, errors="coerce")
    # Opcional: Rellenar fechas faltantes con una fecha por defecto si tu modelo requiere obligatoriamente datos
    # df[date_cols] = df[date_cols].fillna(pd.Timestamp('2026-01-01')) 

    # --- D. Limpieza final de IDs ---
    df["gh_id"] = df["gh_id"].astype(int)
    # Para el ID vinculado, si es NaN, le asignamos 0 para indicar que no tiene un caso pariente
    df["linked_id"] = pd.to_numeric(df["linked_id"], errors="coerce").fillna(0).astype(int)

    return df

# =============================================================================
# EJECUCIÓN DEL SCRIPT
# =============================================================================
if __name__ == "__main__":
    # Reemplaza con la ruta de tu archivo
    PATH_DATASET = "hantavirus2026.csv" 
    
    df_limpio_total = impute_and_clean_dataset(PATH_DATASET)
    
    print("\n### MUESTRA DEL DATASET COMPLETAMENTE SANITIZADO ###")
    print(df_limpio_total[["gh_id", "age", "is_passenger", "status", "sex", "nationality"]].head(10))

-> Columna numérica 'age': NaN reemplazados con la media (63.50)
-> Columna numérica 'confirmed': NaN reemplazados con la media (1.00)
-> Columna de texto 'outcome': NaN reemplazados con la moda ('died')
-> Columna de texto 'sex': NaN reemplazados con la moda ('male')
-> Columna de texto 'nationality': NaN reemplazados con la moda ('british')
-> Columna de texto 'travel_from': NaN reemplazados con la moda ('canary islands')
-> Columna de texto 'left_location': NaN reemplazados con la moda ('st. helena')

### MUESTRA DEL DATASET COMPLETAMENTE SANITIZADO ###
   gh_id   age  is_passenger     status     sex  nationality
0      1  70.0           1.0   probable    male        dutch
1      2  69.0           1.0  confirmed  female        dutch
2      3  63.5           1.0  confirmed    male      british
3      4  63.5           1.0  confirmed  female       german
4      5  63.5           1.0  confirmed    male        swiss
5      6  63.5           0.0   negative  female      british
6      7  